## LLAMA2 FINE TUNING WITH INSTACART DATA

**INSURANCE DATA TRAINING**

In [ ]:
!pip install -q -U trl transformers accelerate git+https://github.com/huggingface/peft.git
!pip install -q datasets bitsandbytes einops wandb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.0/118.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 37.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.2/251.2 kB 27.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 40.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.8/294.8 kB 32.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 93.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 9.

In [ ]:
import pandas as pd
df_joined = pd.read_csv("/content/insurance_training_data_30k.csv")

In [ ]:
df_joined['text'] = df_joined.apply(lambda row: row['input_text'] + " ->: " + row['output_text'], axis = 1)

In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df_joined, test_size=0.2, random_state=42)

In [ ]:
train_df.head(10)

,input_text,output_text,text
24290,Act as an Insurance underwriting expert for AB...,Coverage Details:\n- Policy Name: Comprehensiv...,Act as an Insurance underwriting expert for AB...
19699,Act as an Insurance underwriting expert for AB...,Coverage Details:\n- Premium: $500 per month\n...,Act as an Insurance underwriting expert for AB...
15164,Act as an Insurance underwriting expert for AB...,Coverage Details:\nType of Health Insurance: C...,Act as an Insurance underwriting expert for AB...
13589,Act as an Insurance underwriting expert for AB...,Coverage Details:\nCoverage Type: Comprehensiv...,Act as an Insurance underwriting expert for AB...
25969,Act as an Insurance underwriting expert for AB...,Coverage Details:\nCoverage Level: Silver Plan...,Act as an Insurance underwriting expert for AB...
380,Act as an Insurance underwriting expert for AB...,Coverage Details:\n\nCoverage Areas:\nCoverage...,Act as an Insurance underwriting expert for AB...
32231,Act as an Insurance underwriting expert for AB...,Coverage Details:\n\nCoverage Areas:\n- Hospit...,Act as an Insurance underwriting expert for AB...
12051,Act as an Insurance underwriting expert for AB...,Coverage Details:\nType of Insurance: Health I...,Act as an Insurance underwriting expert for AB...
649,Act as an Insurance underwriting expert for AB...,Coverage Details:\nCoverage Type: Comprehensiv...,Act as an Insurance underwriting expert for AB...
14975,Act as an Insurance underwriting expert for AB...,Coverage Details:\n- Coverage Plan: Comprehens...,Act as an Insurance underwriting expert for AB...


In [ ]:
test_df.head(10)

,input_text,output_text,text
31330,Act as an Insurance underwriting expert for AB...,"Coverage Details:\n- Coverage Limit: $500,000 ...",Act as an Insurance underwriting expert for AB...
2848,Act as an Insurance underwriting expert for AB...,Coverage Details:\nCoverage Type: Health Insur...,Act as an Insurance underwriting expert for AB...
28069,Act as an Insurance underwriting expert for AB...,Coverage Details:**\n- Preferred Provider Orga...,Act as an Insurance underwriting expert for AB...
30623,Act as an Insurance underwriting expert for AB...,Coverage Details:\n- Premium Payment: Monthly\...,Act as an Insurance underwriting expert for AB...
18692,Act as an Insurance underwriting expert for AB...,Coverage Details:\nPremium Amount: $500 per mo...,Act as an Insurance underwriting expert for AB...
28349,Act as an Insurance underwriting expert for AB...,Coverage Details:\nPolicy Term: 1 year\nMonthl...,Act as an Insurance underwriting expert for AB...
20485,Act as an Insurance underwriting expert for AB...,Coverage Details:\nType of Coverage: Comprehen...,Act as an Insurance underwriting expert for AB...
10388,Act as an Insurance underwriting expert for AB...,Coverage Details:\nPremium Amount: $500 per mo...,Act as an Insurance underwriting expert for AB...
3833,Act as an Insurance underwriting expert for AB...,Coverage Details:\nType of Coverage: Health In...,Act as an Insurance underwriting expert for AB...
32083,Act as an Insurance underwriting expert for AB...,Coverage Details:\nCoverage Type: Comprehensiv...,Act as an Insurance underwriting expert for AB...


In [ ]:
from datasets import Dataset,DatasetDict
train_dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df),
})

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer

model_name = "TinyPixel/Llama-2-7B-bf16-sharded"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False

Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import transformers

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # torch_dtype=torch.bfloat16,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto",
)

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Name: Ayesha Nomani
Date of Birth: November 28, 1990
Gender: Female
Address: 123, Sarjapura Road, Bengaluru
Phone Number: +91 987654321
Email: nomani.ayesha@gmail.com
Occupation: Elementary School Teacher

Dependents:
1. Name: Firoz Nomani
Date of Birth: February 22, 1990
Gender: Male
Relationship: Husband

2. Name: Saya Nomani
Date of Birth: June 12, 2019
Relationship: Child

Medical History:
Primary Care Physician: Dr. Jyoti Awasthi
Last Medical Checkup Date: May 11, 2023
Medical Conditions: None

Allergies:
Allergy: Penicillin
Severity: Moderate

Medications:
Medication Name: None
Dosage: N/A
Frequency: N/A

Hospitalizations:
Hospital Name: Springfield Medical Center
Reason: Leg Surgery
Date: June 15, 2015 ->:
"""]

sequences = pipeline(
    test_data,
    max_length=512,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1417: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Name: Ayesha Nomani
Date of Birth: November 28, 1990
Gender: Female
Address: 123, Sarjapura Road, Bengaluru
Phone Number: +91 987654321
Email: nomani.ayesha@gmail.com
Occupation: Elementary School Teacher

Dependents:
1. Name: Firoz Nomani
Date of Birth: February 22, 1990
Gender: Male
Relationship: Husband

2. Name: Saya Nomani
Date of Birth: June 12, 2019
Relationship: Child

Medical History:
Primary Care Physician: Dr. Jyoti Awasthi
Last Medical Checkup Date: May 11, 2023
Medical Conditions: None

Allergies:
Allergy: Penicillin
Severity: Moderate

Medications:
Medication Name: None
Dosage: N/A
Frequency: N/A

Hospitalizations:
Hospital Name: Springfield Medical Center
Reason: Leg Surgery
Date: June 15, 2015 ->:

Hospitalization Details:
- Date: June 15, 2015
- Reason: Leg Surgery
- Hospital Name: Springfield Medical Center
- Doctor: Dr. 

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Jason Pitt
Date of Birth: May 25, 1981
Gender: Male
Address: 753, Richmond Street, California, USA
Phone Number: (555) 682920
Email: jason.pitt218@email.com
Occupation: Software Engineer

Dependents:
1. Name: Emma Pitt
Date of Birth: January 22, 1982
Gender: Female
Relationship: Wife
Dependent Type: Adult

2. Name: Emily Pitt
Date of Birth: June 12, 2005
Relationship: Child
Gender: Female

Medical History:
Primary Care Physician: Dr. Lisa Johnson

Last Medical Checkup Date: May 15, 2023

Medical Conditions: None

Allergies: None

Medications:

Medication Name: None
Dosage: N/A
Frequency: N/A
Hospitalizations:

Hospital Name: Anytown General Hospital
Reason: Appendectomy
Date: March 5, 2019
Surgeries:

Surgery Type: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=512,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Jason Pitt
Date of Birth: May 25, 1981
Gender: Male
Address: 753, Richmond Street, California, USA
Phone Number: (555) 682920
Email: jason.pitt218@email.com
Occupation: Software Engineer

Dependents:
1. Name: Emma Pitt
Date of Birth: January 22, 1982
Gender: Female
Relationship: Wife
Dependent Type: Adult

2. Name: Emily Pitt
Date of Birth: June 12, 2005
Relationship: Child
Gender: Female

Medical History:
Primary Care Physician: Dr. Lisa Johnson

Last Medical Checkup Date: May 15, 2023

Medical Conditions: None

Allergies: None

Medications:

Medication Name: None
Dosage: N/A
Frequency: N/A
Hospitalizations:

Hospital Name: Anytown General Hospital
Reason: Appendectomy
Date: March 5, 2019
Surgeries:

Surgery Type: None ->:
Hospital: None
Reason: None
Date: April 13, 2022

Immunizations:

Vaccine Name: Covid-19
Dose: 1
Date: Jan

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Robert Johnson
Date of Birth: September 5, 1976
Gender: Male
Address: 789 Maple Lane, Riverside, NY
Phone Number: (555) 987-6543
Email: robert.johnson@email.com
Occupation: Accountant

Dependents:

1. Full Name: Olivia Johnson
Date of Birth: December 3, 2005
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Michael Davis
Last Medical Checkup Date: August 12, 2023
Medical Conditions: High Blood Pressure
Allergies: None
Medications:
Medication Name: Lisinopril
Dosage: 10 mg daily
Frequency: Daily
Hospitalizations:

Hospital Name: Riverside General Hospital
Reason: Kidney Stone
Date: July 15, 2010
Surgeries:

Surgery Type: Appendectomy ->:
"""]

sequences = pipeline(
    test_data,
    max_length=512,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Robert Johnson
Date of Birth: September 5, 1976
Gender: Male
Address: 789 Maple Lane, Riverside, NY
Phone Number: (555) 987-6543
Email: robert.johnson@email.com
Occupation: Accountant

Dependents:

1. Full Name: Olivia Johnson
Date of Birth: December 3, 2005
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Michael Davis
Last Medical Checkup Date: August 12, 2023
Medical Conditions: High Blood Pressure
Allergies: None
Medications:
Medication Name: Lisinopril
Dosage: 10 mg daily
Frequency: Daily
Hospitalizations:

Hospital Name: Riverside General Hospital
Reason: Kidney Stone
Date: July 15, 2010
Surgeries:

Surgery Type: Appendectomy ->:
Surgery Name: Appending
Surgeon Name: Dr. James Johnson
Date: May 22, 2010
Procedures:

Procedure Type: Colonoscopy ->:
Procedure Name: Col

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Amisha Dixit
Date of Birth: October 13, 1994
Gender: Female
Address: 1b003, Suncity Gloria, Bangalore, India
Phone Number: (91) 1289024802
Email: amishadixit2738@email.com

Dependents:
1. Full Name: Aarav Dixit
Date of Birth: November 12, 2017
Gender: Male
Relationship: Son
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Jassi Madhav
Last Medical Checkup Date: July 25, 2023
Medical Conditions: None
Allergies: None

Medications:
Medication Name: None
Dosage: N/A
Frequency: N/A

Hospitalizations:
Hospital Name: Lakshmi Medical Center
Reason: None
Date: N/A
Surgeries:

Surgery Type: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=512,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Amisha Dixit
Date of Birth: October 13, 1994
Gender: Female
Address: 1b003, Suncity Gloria, Bangalore, India
Phone Number: (91) 1289024802
Email: amishadixit2738@email.com

Dependents:
1. Full Name: Aarav Dixit
Date of Birth: November 12, 2017
Gender: Male
Relationship: Son
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Jassi Madhav
Last Medical Checkup Date: July 25, 2023
Medical Conditions: None
Allergies: None

Medications:
Medication Name: None
Dosage: N/A
Frequency: N/A

Hospitalizations:
Hospital Name: Lakshmi Medical Center
Reason: None
Date: N/A
Surgeries:

Surgery Type: None ->:
Surgery Reason: None ->:
Surgeon: None
Outpatient Procedures:
Procedure Name: None
Procedure Reason: None <-:
Physician: None

Dental Procedures:
Dentist Name: None
Dental Procedures performed: N/A

Vision Procedures:
Optome

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Lisa Johnson
Date of Birth: July 12, 1972
Gender: Female
Address: 789 Elm Street, Harborview, CA
Phone Number: (555) 987-6543
Email: lisa.johnson@email.com
Occupation: Marketing Manager

Dependents:

1.Full Name: Jack Johnson
Date of Birth: May 20, 2009
Gender: Male
Relationship: Son
Dependent Type: Child

2. Full Name: Jasmine Johnson
Date of Birth: May 20, 2009
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Sarah Lewis
Last Medical Checkup Date: August 28, 2023

Medical Conditions: None

Allergies: None

Medications:

Medication Name: None
Dosage: N/A
Frequency: N/A
Hospitalizations:

Hospital Name: Harborview Medical Center
Reason: None
Date: N/A
Surgeries:

Surgery Type: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=512,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Lisa Johnson
Date of Birth: July 12, 1972
Gender: Female
Address: 789 Elm Street, Harborview, CA
Phone Number: (555) 987-6543
Email: lisa.johnson@email.com
Occupation: Marketing Manager

Dependents:

1.Full Name: Jack Johnson
Date of Birth: May 20, 2009
Gender: Male
Relationship: Son
Dependent Type: Child

2. Full Name: Jasmine Johnson
Date of Birth: May 20, 2009
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Sarah Lewis
Last Medical Checkup Date: August 28, 2023

Medical Conditions: None

Allergies: None

Medications:

Medication Name: None
Dosage: N/A
Frequency: N/A
Hospitalizations:

Hospital Name: Harborview Medical Center
Reason: None
Date: N/A
Surgeries:

Surgery Type: None ->:
Date: N/A
Surgeon: Dr. Sarah Lewis

Medical History Notes:

Notes:


**Note:** These are

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Michelle Anderson
Date of Birth: June 5, 1970
Gender: Female
Address: 123 Oakwood Drive, Green Hills, CA
Phone Number: (555) 123-4567
Email: michelle.anderson@email.com
Occupation: Registered Nurse

Dependents:
Spouse:
Full Name: David Anderson
Date of Birth: August 15, 1968
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Emily Anderson
Date of Birth: November 20, 1995
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Rachel Smith
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Hypertension
Diagnosis Date: January 5, 2010
Treatment: Medication (Lisinopril)
Asthma
Diagnosis Date: August 15, 2000
Treatment: Inhalers (Albuterol and Advair)
Allergies: None

Medications:
Medication Name: Lisinopril (for hypertension)
Dosage: 10 mg daily
Frequency: Daily
Medication Name: Albuterol (for asthma)
Dosage: As needed
Frequency: As needed
Medication Name: Advair (for asthma)
Dosage: As prescribed
Frequency: As prescribed

Hospitalizations:
Hospital Name: Green Hills General Hospital
Reason: Appendectomy
Date: March 20, 1992

Surgeries:

Surgery Type: Appendectomy
Date: March 20, 1992 ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Michelle Anderson
Date of Birth: June 5, 1970
Gender: Female
Address: 123 Oakwood Drive, Green Hills, CA
Phone Number: (555) 123-4567
Email: michelle.anderson@email.com
Occupation: Registered Nurse

Dependents:
Spouse:
Full Name: David Anderson
Date of Birth: August 15, 1968
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Emily Anderson
Date of Birth: November 20, 1995
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Rachel Smith
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Hypertension
Diagnosis Date: January 5, 2010
Treatment: Medication (Lisinopril)
Asthma
Diagnosis Date: August 15, 2000
Treatment: Inhalers (Albuterol and Advair)
Allergies: None

Medications:
Medication Name: Lisinopril (for hypertension)
Dosage: 10 mg daily


In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Rishika Thakur
Date of Birth: November 15, 1990
Gender: Female
Address: 154, Sunpharma Road, Vadodara
Phone Number: +91 8473894725
Email: ririthakur3@email.com
Occupation: Doctor

Dependents:
Spouse:
Full Name: Dhruv Thakur
Date of Birth: June 21, 1988
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Esha Thakur
Date of Birth: November 12, 2020
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:

Primary Care Physician: Dr. Rakhi Sharma
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Type 2 Diabetes
Diagnosis Date: June 10, 2015
Treatment: Insulin and Metformin
Arthritis
Diagnosis Date: November 8, 2005
Treatment: Nonsteroidal Anti-Inflammatory Drugs (NSAIDs)
Allergies: None

Medications:
Medication Name: Insulin (for diabetes)
Dosage: As prescribed
Frequency: As prescribed
Medication Name: Metformin (for diabetes)
Dosage: 500 mg twice daily
Frequency: Daily
Medication Name: Ibuprofen (for arthritis)
Dosage: 400 mg as needed
Frequency: As needed
Hospitalizations:

Hospital Name: Green Hills General Hospital
Reason: Knee Replacement Surgery
Date: May 8, 2018
Hospital Name: Green Hills General Hospital
Reason: Gallbladder Removal Surgery
Date: June 15, 2022
Surgeries:

Surgery Type: Knee Replacement Surgery
Date: May 8, 2018
Surgery Type: Gallbladder Removal Surgery
Date: June 15, 2022 ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Rishika Thakur
Date of Birth: November 15, 1990
Gender: Female
Address: 154, Sunpharma Road, Vadodara
Phone Number: +91 8473894725
Email: ririthakur3@email.com
Occupation: Doctor

Dependents:
Spouse:
Full Name: Dhruv Thakur
Date of Birth: June 21, 1988
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Esha Thakur
Date of Birth: November 12, 2020
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:

Primary Care Physician: Dr. Rakhi Sharma
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Type 2 Diabetes
Diagnosis Date: June 10, 2015
Treatment: Insulin and Metformin
Arthritis
Diagnosis Date: November 8, 2005
Treatment: Nonsteroidal Anti-Inflammatory Drugs (NSAIDs)
Allergies: None

Medications:
Medication Name: Insulin (for diabetes)
Dosage: As prescribed
Frequency: As

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Kevin Walker
Date of Birth: April 30, 1985
Gender: Male
Address: 456 Maple Street, Riverside, FL
Phone Number: (555) 789-1234
Email: kevin.walker@email.com
Occupation: Electrician

Dependents:

Spouse:
Full Name: Jessica Walker
Date of Birth: September 20, 1987
Gender: Female
Relationship: Wife
Dependent Type: Adult

Medical History:
Primary Care Physician: Dr. Thomas Brown
Last Medical Checkup Date: August 10, 2023
Medical Conditions:
Chronic Back Pain
Diagnosis Date: July 5, 2015
Treatment: Physical Therapy and Pain Medication
Migraine Headaches
Diagnosis Date: April 15, 2007
Treatment: Triptans and Lifestyle Management
Allergies: None

Medications:
Medication Name: Naproxen (for back pain)
Dosage: 500 mg twice daily
Frequency: Daily
Medication Name: Sumatriptan (for migraines)
Dosage: 100 mg as needed
Frequency: As needed
Hospitalizations 8:

Hospital Name: Riverside Medical Center
Reason: Appendectomy
Date: February 18, 2010

Surgeries:
None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1417: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Kevin Walker
Date of Birth: April 30, 1985
Gender: Male
Address: 456 Maple Street, Riverside, FL
Phone Number: (555) 789-1234
Email: kevin.walker@email.com
Occupation: Electrician

Dependents:

Spouse:
Full Name: Jessica Walker
Date of Birth: September 20, 1987
Gender: Female
Relationship: Wife
Dependent Type: Adult

Medical History:
Primary Care Physician: Dr. Thomas Brown
Last Medical Checkup Date: August 10, 2023
Medical Conditions:
Chronic Back Pain
Diagnosis Date: July 5, 2015
Treatment: Physical Therapy and Pain Medication
Migraine Headaches
Diagnosis Date: April 15, 2007
Treatment: Triptans and Lifestyle Management
Allergies: None

Medications:
Medication Name: Naproxen (for back pain)
Dosage: 500 mg twice daily
Frequency: Daily
Medication Name: Sumatriptan (for migraines)
Dosage: 100 mg as needed
Frequency: As needed
Hos

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Lisa Martinez
Date of Birth: October 12, 1982
Gender: Female
Address: 789 Cedar Avenue, Oakville, NY
Phone Number: (555) 123-4567
Email: lisa.martinez@email.com
Occupation: Pediatrician

Dependents:
Spouse:
Full Name: Carlos Martinez
Date of Birth: December 5, 1980
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Name: Ashley Martinez
Date of Birth: August 23 2010
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Jennifer Adams
Last Medical Checkup Date: September 5, 2023
Medical Conditions:
Asthma
Diagnosis Date: January 10, 1990
Treatment: Inhalers (Albuterol and Advair)
Allergic Rhinitis (Hay Fever)
Diagnosis Date: April 15, 2005
Treatment: Antihistamines
Allergies:
Allergy: Penicillin
Severity: Moderate

Medications:
Medication Name: Albuterol (for asthma)
Dosage: As needed
Frequency: As needed
Medication Name: Advair (for asthma)
Dosage: As prescribed
Frequency: As prescribed
Medication Name: Loratadine (for allergies)
Dosage: 10 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Oakville General Hospital
Reason: Kidney Stone
Date: N/A

Surgeries:
Surgery Type: Kidney Stone Removal
Date of Admission: January 19 2007 ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Lisa Martinez
Date of Birth: October 12, 1982
Gender: Female
Address: 789 Cedar Avenue, Oakville, NY
Phone Number: (555) 123-4567
Email: lisa.martinez@email.com
Occupation: Pediatrician

Dependents:
Spouse:
Full Name: Carlos Martinez
Date of Birth: December 5, 1980
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Name: Ashley Martinez
Date of Birth: August 23 2010
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Jennifer Adams
Last Medical Checkup Date: September 5, 2023
Medical Conditions:
Asthma
Diagnosis Date: January 10, 1990
Treatment: Inhalers (Albuterol and Advair)
Allergic Rhinitis (Hay Fever)
Diagnosis Date: April 15, 2005
Treatment: Antihistamines
Allergies:
Allergy: Penicillin
Severity: Moderate

Medications:
Medication Name: Albuterol (for asthm

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Daniel Lee
Date of Birth: July 20, 1978
Gender: Male
Address: 321 Elm Street, Pineville, TX
Phone Number: (555) 987-6543
Email: daniel.lee@email.com
Occupation: Architect

Dependents:
Spouse:
Full Name: Laura Lee
Date of Birth: March 10, 1982
Gender: Female
Relationship: Wife
Dependent Type: Adult

Child 1:
Full Name: Ethan Lee
Date of Birth: December 15, 2010
Gender: Male
Relationship: Son
Dependent Type: Child

Child 2:
Full Name: Ema Lee
Date of Birth: April 06, 2007
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Susan Wilson
Last Medical Checkup Date: July 10, 2023
Medical Conditions:
Type 2 Diabetes
Diagnosis Date: May 5, 2012
Treatment: Metformin and Dietary Management
High Cholesterol (Hypercholesterolemia)
Diagnosis Date: October 20, 2018
Treatment: Statins (Atorvastatin)
Allergies: None

Medications:
Medication Name: Metformin (for diabetes)
Dosage: 500 mg twice daily
Frequency: Daily
Medication Name: Atorvastatin (for high cholesterol)
Dosage: 20 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Pineville Medical Center
Reason: Fever
Date: N/A

Surgeries:
Surgery Type: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Daniel Lee
Date of Birth: July 20, 1978
Gender: Male
Address: 321 Elm Street, Pineville, TX
Phone Number: (555) 987-6543
Email: daniel.lee@email.com
Occupation: Architect

Dependents:
Spouse:
Full Name: Laura Lee
Date of Birth: March 10, 1982
Gender: Female
Relationship: Wife
Dependent Type: Adult

Child 1:
Full Name: Ethan Lee
Date of Birth: December 15, 2010
Gender: Male
Relationship: Son
Dependent Type: Child

Child 2:
Full Name: Ema Lee
Date of Birth: April 06, 2007
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Susan Wilson
Last Medical Checkup Date: July 10, 2023
Medical Conditions:
Type 2 Diabetes
Diagnosis Date: May 5, 2012
Treatment: Metformin and Dietary Management
High Cholesterol (Hypercholesterolemia)
Diagnosis Date: October 20, 2018
Treatment: Statins (Ator

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Rachel Turner
Date of Birth: November 8, 1990
Gender: Female
Address: 789 Birch Road, Harborview, CA
Phone Number: (555) 123-7890
Email: rachel.turner@email.com
Occupation: Software Developer

Dependents:
Spouse:
Full Name: Matthew Turner
Date of Birth: April 15, 1992
Gender: Male
Relationship: Husband
Dependent Type: Adult


Medical History:
Rachel Turner
Primary Care Physician: Dr. Emily Harris
Last Medical Checkup Date: August 15, 2023
Medical Conditions:
Anxiety Disorder
Diagnosis Date: June 5, 2015
Treatment: Cognitive-Behavioral Therapy and Medication (SSRIs)
Seasonal Affective Disorder (SAD)
Diagnosis Date: December 10, 2012
Treatment: Light Therapy and Lifestyle Management
Allergies: None

Medications:
Medication Name: Sertraline (for anxiety)
Dosage: 50 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Harborview General Hospital
Reason: None
Date: N/A

Surgeries:

Surgery Type: None

Matthew Turner
Primary Care Physician: Dr. Emily Harris
Last Medical Checkup Date: August 15, 2023
- No pre-existing medical conditions
- Hospitalized for pneumonia in 2016, fully recovered
- No chronic illnesses or disabilities ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Rachel Turner
Date of Birth: November 8, 1990
Gender: Female
Address: 789 Birch Road, Harborview, CA
Phone Number: (555) 123-7890
Email: rachel.turner@email.com
Occupation: Software Developer

Dependents:
Spouse:
Full Name: Matthew Turner
Date of Birth: April 15, 1992
Gender: Male
Relationship: Husband
Dependent Type: Adult


Medical History:
Rachel Turner
Primary Care Physician: Dr. Emily Harris
Last Medical Checkup Date: August 15, 2023
Medical Conditions:
Anxiety Disorder
Diagnosis Date: June 5, 2015
Treatment: Cognitive-Behavioral Therapy and Medication (SSRIs)
Seasonal Affective Disorder (SAD)
Diagnosis Date: December 10, 2012
Treatment: Light Therapy and Lifestyle Management
Allergies: None

Medications:
Medication Name: Sertraline (for anxiety)
Dosage: 50 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Harborv

In [ ]:
## BEFORE FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Samuel Harris
Date of Birth: March 25, 1986
Gender: Male
Address: 456 Spruce Street, Woodville, TX
Phone Number: (555) 456-7891
Email: samuel.harris@email.com
Occupation: Accountant

Dependents:
Spouse:
Full Name: Olivia Harris
Date of Birth: October 20, 1988
Gender: Female
Relationship: Wife
Dependent Type: Adult

Child:
Full Name: Phoebe Harris
Date of Birth: December 12, 2012
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Samuel Harris
Primary Care Physician: Dr. Michael Turner
Last Medical Checkup Date: September 20, 2023
Medical Conditions:
Asthma
Diagnosis Date: January 5, 1995
Treatment: Inhalers (Albuterol and Advair)
GERD (Gastroesophageal Reflux Disease)
Diagnosis Date: April 15, 2010
Treatment: Proton Pump Inhibitors (PPIs)
Allergies:
Allergy: Shellfish
Severity: Moderate
Medications 12:

Medication Name: Albuterol (for asthma)
Dosage: As needed
Frequency: As needed
Medication Name: Advair (for asthma)
Dosage: As prescribed
Frequency: As prescribed
Medication Name: Omeprazole (for GERD)
Dosage: 20 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Woodville Regional Hospital
Reason: Asthma
Date: February 22, 2015

Surgeries:
Surgery Type: None

Olivia Harris
Primary Care Physician: Dr. Michael Turner
Last Medical Checkup Date: March 18, 2023

Medical Conditions:
Anxiety Disorder
Diagnosis Date: June 5, 2015
Treatment: Cognitive-Behavioral Therapy and Medication (SSRIs)

Medications:
Medication Name: Sertraline (for anxiety)
Dosage: 50 mg daily
Frequency: Daily

Phoebe Harris:
- No major health conditions or surgeries in the past 5 years
- Prescription medications: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Samuel Harris
Date of Birth: March 25, 1986
Gender: Male
Address: 456 Spruce Street, Woodville, TX
Phone Number: (555) 456-7891
Email: samuel.harris@email.com
Occupation: Accountant

Dependents:
Spouse:
Full Name: Olivia Harris
Date of Birth: October 20, 1988
Gender: Female
Relationship: Wife
Dependent Type: Adult

Child:
Full Name: Phoebe Harris
Date of Birth: December 12, 2012
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Samuel Harris
Primary Care Physician: Dr. Michael Turner
Last Medical Checkup Date: September 20, 2023
Medical Conditions:
Asthma
Diagnosis Date: January 5, 1995
Treatment: Inhalers (Albuterol and Advair)
GERD (Gastroesophageal Reflux Disease)
Diagnosis Date: April 15, 2010
Treatment: Proton Pump Inhibitors (PPIs)
Allergies:
Allergy: Shellfish
Severity: Moderate
Medications 12:

**FINETUNING THE MODEL**

In [ ]:
from peft import LoraConfig

lora_alpha = 16
lora_dropout = 0.1
lora_r = 64

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","v_proj"]
)

In [ ]:
from transformers import TrainingArguments

output_dir = "./insurance_results"
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
optim = "paged_adamw_32bit"
save_steps = 10
logging_steps = 1
learning_rate = 2e-4
max_grad_norm = 0.3
max_steps = 120
warmup_ratio = 0.03
lr_scheduler_type = "constant"

training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
)

In [ ]:
from trl import SFTTrainer

max_seq_length = 512

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_dict['train'],
    # train_dataset=data['train'],
    peft_config=peft_config,
    dataset_text_field="text",
    # dataset_text_field="prediction",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
)

/usr/local/lib/python3.10/dist-packages/peft/utils/other.py:133: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(


Map:   0%|          | 0/26120 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:207: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


In [ ]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [ ]:
for name, module in trainer.model.named_modules():
    if "norm" in name:
        module = module.to(torch.float32)

In [ ]:
! wandb login 942c0bdca2f1b19b6cef0464f048c21bfdf93c24

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [ ]:
trainer.train()

wandb: Currently logged in as: stuti-srivastava. Use `wandb login --relogin` to force relogin


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
1,1.093900
2,1.029300
3,1.040700
4,1.004900
5,0.963200
6,0.934700
7,0.893600
8,0.871300
9,0.834700
10,0.822100


TrainOutput(global_step=120, training_loss=0.43955542674909037, metrics={'train_runtime': 2963.9943, 'train_samples_per_second': 0.648, 'train_steps_per_second': 0.04, 'total_flos': 1.94467302875136e+16, 'train_loss': 0.43955542674909037, 'epoch': 0.07})

In [35]:
test_df['test_text'] = test_df.apply(lambda row: row['input_text'] + " ->: ", axis = 1)
lst_test_data = list(test_df['test_text'])

In [36]:
len(lst_test_data)

6531

In [37]:
sample_size = 10
lst_test_data_short = lst_test_data[:sample_size]

In [ ]:
import transformers

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # torch_dtype=torch.bfloat16,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto",
)

In [38]:
sequences = pipeline(
    lst_test_data_short,
    max_length=1024,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:31: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn("None of the inputs have requires_grad=True. Gradients will be None")


KeyboardInterrupt: ignored

In [ ]:
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Name: Ayesha Nomani
Date of Birth: November 28, 1990
Gender: Female
Address: 123, Sarjapura Road, Bengaluru
Phone Number: +91 987654321
Email: nomani.ayesha@gmail.com
Occupation: Elementary School Teacher

Dependents:
1. Name: Firoz Nomani
Date of Birth: February 22, 1990
Gender: Male
Relationship: Husband

2. Name: Saya Nomani
Date of Birth: June 12, 2019
Relationship: Child

Medical History:
Primary Care Physician: Dr. Jyoti Awasthi
Last Medical Checkup Date: May 11, 2023
Medical Conditions: None

Allergies:
Allergy: Penicillin
Severity: Moderate

Medications:
Medication Name: None
Dosage: N/A
Frequency: N/A

Hospitalizations:
Hospital Name: Springfield Medical Center
Reason: Leg Surgery
Date: June 15, 2015

Firoz Nomani
- Asthma diagnosis in 2010, but well-controlled with medication
- Non-smoker
- No history of serious illnesses

Saya Nomani
- Overall good health
- No known allergies or pre-existing medical conditions
- Non-smoker
- No history of serious illnesses
->:
"""]

sequences = pipeline(
    test_data,
    max_length=512,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])

0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Name: Ayesha Nomani
Date of Birth: November 28, 1990
Gender: Female
Address: 123, Sarjapura Road, Bengaluru
Phone Number: +91 987654321
Email: nomani.ayesha@gmail.com
Occupation: Elementary School Teacher

Dependents:
1. Name: Firoz Nomani
Date of Birth: February 22, 1990
Gender: Male
Relationship: Husband

2. Name: Saya Nomani
Date of Birth: June 12, 2019
Relationship: Child

Medical History:
Primary Care Physician: Dr. Jyoti Awasthi
Last Medical Checkup Date: May 11, 2023
Medical Conditions: None

Allergies:
Allergy: Penicillin
Severity: Moderate

Medications:
Medication Name: None
Dosage: N/A
Frequency: N/A

Hospitalizations:
Hospital Name: Springfield Medical Center
Reason: Leg Surgery
Date: June 15, 2015 ->:
Hospitalization Details:
- Admission Date: June 15, 2015
- Discharge Date: June 21, 2015
- Length of Stay (in Days): 6
- Reason 

In [ ]:
## AFTER FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Jason Pitt
Date of Birth: May 25, 1981
Gender: Male
Address: 753, Richmond Street, California, USA
Phone Number: (555) 682920
Email: jason.pitt218@email.com
Occupation: Software Engineer

Dependents:
1. Name: Emma Pitt
Date of Birth: January 22, 1982
Gender: Female
Relationship: Wife
Dependent Type: Adult

2. Name: Emily Pitt
Date of Birth: June 12, 2005
Relationship: Child
Gender: Female

Medical History:
Primary Care Physician: Dr. Lisa Johnson

Last Medical Checkup Date: May 15, 2023

Medical Conditions: None

Allergies: None

Medications:

Medication Name: None
Dosage: N/A
Frequency: N/A
Hospitalizations:

Hospital Name: Anytown General Hospital
Reason: Appendectomy
Date: March 5, 2019
Surgeries:

Surgery Type: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Jason Pitt
Date of Birth: May 25, 1981
Gender: Male
Address: 753, Richmond Street, California, USA
Phone Number: (555) 682920
Email: jason.pitt218@email.com
Occupation: Software Engineer

Dependents:
1. Name: Emma Pitt
Date of Birth: January 22, 1982
Gender: Female
Relationship: Wife
Dependent Type: Adult

2. Name: Emily Pitt
Date of Birth: June 12, 2005
Relationship: Child
Gender: Female

Medical History:
Primary Care Physician: Dr. Lisa Johnson

Last Medical Checkup Date: May 15, 2023

Medical Conditions: None

Allergies: None

Medications:

Medication Name: None
Dosage: N/A
Frequency: N/A
Hospitalizations:

Hospital Name: Anytown General Hospital
Reason: Appendectomy
Date: March 5, 2019
Surgeries:

Surgery Type: None ->:
Name of Surgeon: Dr. Smith
Hospital Name: None
Date: June 17, 2022
Hospitalizations:

Hospital Name: Anyto

In [ ]:
## AFTER FINETUNING
test_data = ["""0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Michelle Anderson
Date of Birth: June 5, 1970
Gender: Female
Address: 123 Oakwood Drive, Green Hills, CA
Phone Number: (555) 123-4567
Email: michelle.anderson@email.com
Occupation: Registered Nurse

Dependents:
Spouse:
Full Name: David Anderson
Date of Birth: August 15, 1968
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Emily Anderson
Date of Birth: November 20, 1995
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Rachel Smith
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Hypertension
Diagnosis Date: January 5, 2010
Treatment: Medication (Lisinopril)
Asthma
Diagnosis Date: August 15, 2000
Treatment: Inhalers (Albuterol and Advair)
Allergies: None

Medications:
Medication Name: Lisinopril (for hypertension)
Dosage: 10 mg daily
Frequency: Daily
Medication Name: Albuterol (for asthma)
Dosage: As needed
Frequency: As needed
Medication Name: Advair (for asthma)
Dosage: As prescribed
Frequency: As prescribed

Hospitalizations:
Hospital Name: Green Hills General Hospital
Reason: Appendectomy
Date: March 20, 1992

Surgeries:

Surgery Type: Appendectomy
Date: March 20, 1992 ->:
"""]

sequences = pipeline(
    test_data,
    max_length=1024,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:31: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn("None of the inputs have requires_grad=True. Gradients will be None")


0 0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Michelle Anderson
Date of Birth: June 5, 1970
Gender: Female
Address: 123 Oakwood Drive, Green Hills, CA
Phone Number: (555) 123-4567
Email: michelle.anderson@email.com
Occupation: Registered Nurse

Dependents:
Spouse:
Full Name: David Anderson
Date of Birth: August 15, 1968
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Emily Anderson
Date of Birth: November 20, 1995
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Rachel Smith
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Hypertension
Diagnosis Date: January 5, 2010
Treatment: Medication (Lisinopril)
Asthma
Diagnosis Date: August 15, 2000
Treatment: Inhalers (Albuterol and Advair)
Allergies: None

Medications:
Medication Name: Lisinopril (for hypertension)
Dosage: 10 mg dail

In [ ]:
## AFTER FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Robert Johnson
Date of Birth: September 5, 1976
Gender: Male
Address: 789 Maple Lane, Riverside, NY
Phone Number: (555) 987-6543
Email: robert.johnson@email.com
Occupation: Accountant

Dependents:

1. Full Name: Olivia Johnson
Date of Birth: December 3, 2005
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Michael Davis
Last Medical Checkup Date: August 12, 2023
Medical Conditions: High Blood Pressure
Allergies: None
Medications:
Medication Name: Lisinopril
Dosage: 10 mg daily
Frequency: Daily
Hospitalizations:

Hospital Name: Riverside General Hospital
Reason: Kidney Stone
Date: July 15, 2010
Surgeries:

Surgery Type: Appendectomy ->:
"""]

sequences = pipeline(
    test_data,
    max_length=1024,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Robert Johnson
Date of Birth: September 5, 1976
Gender: Male
Address: 789 Maple Lane, Riverside, NY
Phone Number: (555) 987-6543
Email: robert.johnson@email.com
Occupation: Accountant

Dependents:

1. Full Name: Olivia Johnson
Date of Birth: December 3, 2005
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Michael Davis
Last Medical Checkup Date: August 12, 2023
Medical Conditions: High Blood Pressure
Allergies: None
Medications:
Medication Name: Lisinopril
Dosage: 10 mg daily
Frequency: Daily
Hospitalizations:

Hospital Name: Riverside General Hospital
Reason: Kidney Stone
Date: July 15, 2010
Surgeries:

Surgery Type: Appendectomy ->:
Date: March 5, 1997
Hospital: Riverside General Hospital
Surgeon: Dr. Robert Jones

Surgery Type: Carpal Tunnel Surgery ->:
Date: January 1

In [ ]:
## AFTER FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Amisha Dixit
Date of Birth: October 13, 1994
Gender: Female
Address: 1b003, Suncity Gloria, Bangalore, India
Phone Number: (91) 1289024802
Email: amishadixit2738@email.com

Dependents:
1. Full Name: Aarav Dixit
Date of Birth: November 12, 2017
Gender: Male
Relationship: Son
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Jassi Madhav
Last Medical Checkup Date: July 25, 2023
Medical Conditions: None
Allergies: None

Medications:
Medication Name: None
Dosage: N/A
Frequency: N/A

Hospitalizations:
Hospital Name: Lakshmi Medical Center
Reason: None
Date: N/A
Surgeries:

Surgery Type: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Amisha Dixit
Date of Birth: October 13, 1994
Gender: Female
Address: 1b003, Suncity Gloria, Bangalore, India
Phone Number: (91) 1289024802
Email: amishadixit2738@email.com

Dependents:
1. Full Name: Aarav Dixit
Date of Birth: November 12, 2017
Gender: Male
Relationship: Son
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Jassi Madhav
Last Medical Checkup Date: July 25, 2023
Medical Conditions: None
Allergies: None

Medications:
Medication Name: None
Dosage: N/A
Frequency: N/A

Hospitalizations:
Hospital Name: Lakshmi Medical Center
Reason: None
Date: N/A
Surgeries:

Surgery Type: None ->:
Date: N/A
Reason: None

Insurance Policy Details:

Type of Policy: Comprehensive Health Insurance
Plan Level: Gold

Policy Details:
- Premium Amount: $300 per month
- Deductible Amount: $1,000
- Co-payment Amount: $40 for Pr

In [ ]:
## AFTER FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Lisa Johnson
Date of Birth: July 12, 1972
Gender: Female
Address: 789 Elm Street, Harborview, CA
Phone Number: (555) 987-6543
Email: lisa.johnson@email.com
Occupation: Marketing Manager

Dependents:

1.Full Name: Jack Johnson
Date of Birth: May 20, 2009
Gender: Male
Relationship: Son
Dependent Type: Child

2. Full Name: Jasmine Johnson
Date of Birth: May 20, 2009
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Sarah Lewis
Last Medical Checkup Date: August 28, 2023

Medical Conditions: None

Allergies: None

Medications:

Medication Name: None
Dosage: N/A
Frequency: N/A
Hospitalizations:

Hospital Name: Harborview Medical Center
Reason: None
Date: N/A
Surgeries:

Surgery Type: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Lisa Johnson
Date of Birth: July 12, 1972
Gender: Female
Address: 789 Elm Street, Harborview, CA
Phone Number: (555) 987-6543
Email: lisa.johnson@email.com
Occupation: Marketing Manager

Dependents:

1.Full Name: Jack Johnson
Date of Birth: May 20, 2009
Gender: Male
Relationship: Son
Dependent Type: Child

2. Full Name: Jasmine Johnson
Date of Birth: May 20, 2009
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Sarah Lewis
Last Medical Checkup Date: August 28, 2023

Medical Conditions: None

Allergies: None

Medications:

Medication Name: None
Dosage: N/A
Frequency: N/A
Hospitalizations:

Hospital Name: Harborview Medical Center
Reason: None
Date: N/A
Surgeries:

Surgery Type: None ->:
Reason:
Date: N/A

 ->:

- Policy Details:
Annual Premium: $4,000
Deductible: $1,500
Co-

In [ ]:
## AFTER FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Michelle Anderson
Date of Birth: June 5, 1970
Gender: Female
Address: 123 Oakwood Drive, Green Hills, CA
Phone Number: (555) 123-4567
Email: michelle.anderson@email.com
Occupation: Registered Nurse

Dependents:
Spouse:
Full Name: David Anderson
Date of Birth: August 15, 1968
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Emily Anderson
Date of Birth: November 20, 1995
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Rachel Smith
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Hypertension
Diagnosis Date: January 5, 2010
Treatment: Medication (Lisinopril)
Asthma
Diagnosis Date: August 15, 2000
Treatment: Inhalers (Albuterol and Advair)
Allergies: None

Medications:
Medication Name: Lisinopril (for hypertension)
Dosage: 10 mg daily
Frequency: Daily
Medication Name: Albuterol (for asthma)
Dosage: As needed
Frequency: As needed
Medication Name: Advair (for asthma)
Dosage: As prescribed
Frequency: As prescribed

Hospitalizations:
Hospital Name: Green Hills General Hospital
Reason: Appendectomy
Date: March 20, 1992

Surgeries:

Surgery Type: Appendectomy
Date: March 20, 1992
->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Michelle Anderson
Date of Birth: June 5, 1970
Gender: Female
Address: 123 Oakwood Drive, Green Hills, CA
Phone Number: (555) 123-4567
Email: michelle.anderson@email.com
Occupation: Registered Nurse

Dependents:
Spouse:
Full Name: David Anderson
Date of Birth: August 15, 1968
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Emily Anderson
Date of Birth: November 20, 1995
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Rachel Smith
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Hypertension
Diagnosis Date: January 5, 2010
Treatment: Medication (Lisinopril)
Asthma
Diagnosis Date: August 15, 2000
Treatment: Inhalers (Albuterol and Advair)
Allergies: None

Medications:
Medication Name: Lisinopril (for hypertension)
Dosage: 10 mg daily


In [ ]:
## AFTER FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Rishika Thakur
Date of Birth: November 15, 1990
Gender: Female
Address: 154, Sunpharma Road, Vadodara
Phone Number: +91 8473894725
Email: ririthakur3@email.com
Occupation: Doctor

Dependents:
Spouse:
Full Name: Dhruv Thakur
Date of Birth: June 21, 1988
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Esha Thakur
Date of Birth: November 12, 2020
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:

Primary Care Physician: Dr. Rakhi Sharma
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Type 2 Diabetes
Diagnosis Date: June 10, 2015
Treatment: Insulin and Metformin
Arthritis
Diagnosis Date: November 8, 2005
Treatment: Nonsteroidal Anti-Inflammatory Drugs (NSAIDs)
Allergies: None

Medications:
Medication Name: Insulin (for diabetes)
Dosage: As prescribed
Frequency: As prescribed
Medication Name: Metformin (for diabetes)
Dosage: 500 mg twice daily
Frequency: Daily
Medication Name: Ibuprofen (for arthritis)
Dosage: 400 mg as needed
Frequency: As needed
Hospitalizations:

Hospital Name: Green Hills General Hospital
Reason: Knee Replacement Surgery
Date: May 8, 2018
Hospital Name: Green Hills General Hospital
Reason: Gallbladder Removal Surgery
Date: June 15, 2022
Surgeries:

Surgery Type: Knee Replacement Surgery
Date: May 8, 2018
Surgery Type: Gallbladder Removal Surgery
Date: June 15, 2022 ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Rishika Thakur
Date of Birth: November 15, 1990
Gender: Female
Address: 154, Sunpharma Road, Vadodara
Phone Number: +91 8473894725
Email: ririthakur3@email.com
Occupation: Doctor

Dependents:
Spouse:
Full Name: Dhruv Thakur
Date of Birth: June 21, 1988
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Esha Thakur
Date of Birth: November 12, 2020
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:

Primary Care Physician: Dr. Rakhi Sharma
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Type 2 Diabetes
Diagnosis Date: June 10, 2015
Treatment: Insulin and Metformin
Arthritis
Diagnosis Date: November 8, 2005
Treatment: Nonsteroidal Anti-Inflammatory Drugs (NSAIDs)
Allergies: None

Medications:
Medication Name: Insulin (for diabetes)
Dosage: As prescribed
Frequency: As

In [27]:
## AFTER FINETUNING
test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Kevin Walker
Date of Birth: April 30, 1985
Gender: Male
Address: 456 Maple Street, Riverside, FL
Phone Number: (555) 789-1234
Email: kevin.walker@email.com
Occupation: Electrician

Dependents:
Spouse:
Full Name: Jessica Walker
Date of Birth: September 20, 1987
Gender: Female
Relationship: Wife
Dependent Type: Adult

Medical History:
Primary Care Physician: Dr. Thomas Brown
Last Medical Checkup Date: August 10, 2023
Medical Conditions:
Chronic Back Pain
Diagnosis Date: July 5, 2015
Treatment: Physical Therapy and Pain Medication
Migraine Headaches
Diagnosis Date: April 15, 2007
Treatment: Triptans and Lifestyle Management
Allergies: None

Medications:
Medication Name: Naproxen (for back pain)
Dosage: 500 mg twice daily
Frequency: Daily
Medication Name: Sumatriptan (for migraines)
Dosage: 100 mg as needed
Frequency: As needed

Hospitalizations:
Hospital Name: Riverside Medical Center
Reason: Appendectomy
Date: February 18, 2010

Surgeries:
None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Kevin Walker
Date of Birth: April 30, 1985
Gender: Male
Address: 456 Maple Street, Riverside, FL
Phone Number: (555) 789-1234
Email: kevin.walker@email.com
Occupation: Electrician

Dependents:
Spouse:
Full Name: Jessica Walker
Date of Birth: September 20, 1987
Gender: Female
Relationship: Wife
Dependent Type: Adult

Medical History:
Primary Care Physician: Dr. Thomas Brown
Last Medical Checkup Date: August 10, 2023
Medical Conditions:
Chronic Back Pain
Diagnosis Date: July 5, 2015
Treatment: Physical Therapy and Pain Medication
Migraine Headaches
Diagnosis Date: April 15, 2007
Treatment: Triptans and Lifestyle Management
Allergies: None

Medications:
Medication Name: Naproxen (for back pain)
Dosage: 500 mg twice daily
Frequency: Daily
Medication Name: Sumatriptan (for migraines)
Dosage: 100 mg as needed
Frequency: As needed

Hos

In [29]:
## AFTER FINETUNING
test_data = ["""
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Lisa Martinez
Date of Birth: October 12, 1982
Gender: Female
Address: 789 Cedar Avenue, Oakville, NY
Phone Number: (555) 123-4567
Email: lisa.martinez@email.com
Occupation: Pediatrician

Dependents:
Spouse:
Full Name: Carlos Martinez
Date of Birth: December 5, 1980
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Name: Ashley Martinez
Date of Birth: August 23 2010
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Jennifer Adams
Last Medical Checkup Date: September 5, 2023
Medical Conditions:
Asthma
Diagnosis Date: January 10, 1990
Treatment: Inhalers (Albuterol and Advair)
Allergic Rhinitis (Hay Fever)
Diagnosis Date: April 15, 2005
Treatment: Antihistamines
Allergies:
Allergy: Penicillin
Severity: Moderate

Medications:
Medication Name: Albuterol (for asthma)
Dosage: As needed
Frequency: As needed
Medication Name: Advair (for asthma)
Dosage: As prescribed
Frequency: As prescribed
Medication Name: Loratadine (for allergies)
Dosage: 10 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Oakville General Hospital
Reason: Kidney Stone
Date: N/A

Surgeries:
Surgery Type: Kidney Stone Removal
Date of Admission: January 19 2007 ->:
"""]

sequences = pipeline(
    test_data,
    max_length=800,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])

0 
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Lisa Martinez
Date of Birth: October 12, 1982
Gender: Female
Address: 789 Cedar Avenue, Oakville, NY
Phone Number: (555) 123-4567
Email: lisa.martinez@email.com
Occupation: Pediatrician

Dependents:
Spouse:
Full Name: Carlos Martinez
Date of Birth: December 5, 1980
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Name: Ashley Martinez
Date of Birth: August 23 2010
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Jennifer Adams
Last Medical Checkup Date: September 5, 2023
Medical Conditions:
Asthma
Diagnosis Date: January 10, 1990
Treatment: Inhalers (Albuterol and Advair)
Allergic Rhinitis (Hay Fever)
Diagnosis Date: April 15, 2005
Treatment: Antihistamines
Allergies:
Allergy: Penicillin
Severity: Moderate

Medications:
Medication Name: Albuterol (for asth

In [33]:
## AFTER FINETUNING
test_data = ["""
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Daniel Lee
Date of Birth: July 20, 1978
Gender: Male
Address: 321 Elm Street, Pineville, TX
Phone Number: (555) 987-6543
Email: daniel.lee@email.com
Occupation: Architect

Dependents:
Spouse:
Full Name: Laura Lee
Date of Birth: March 10, 1982
Gender: Female
Relationship: Wife
Dependent Type: Adult

Child 1:
Full Name: Ethan Lee
Date of Birth: December 15, 2010
Gender: Male
Relationship: Son
Dependent Type: Child

Child 2:
Full Name: Ema Lee
Date of Birth: April 06, 2007
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Susan Wilson
Last Medical Checkup Date: July 10, 2023
Medical Conditions:
Type 2 Diabetes
Diagnosis Date: May 5, 2012
Treatment: Metformin and Dietary Management
High Cholesterol (Hypercholesterolemia)
Diagnosis Date: October 20, 2018
Treatment: Statins (Atorvastatin)
Allergies: None

Medications:
Medication Name: Metformin (for diabetes)
Dosage: 500 mg twice daily
Frequency: Daily
Medication Name: Atorvastatin (for high cholesterol)
Dosage: 20 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Pineville Medical Center
Reason: Fever
Date: N/A

Surgeries:
Surgery Type: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=1024,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])

0 
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Daniel Lee
Date of Birth: July 20, 1978
Gender: Male
Address: 321 Elm Street, Pineville, TX
Phone Number: (555) 987-6543
Email: daniel.lee@email.com
Occupation: Architect

Dependents:
Spouse:
Full Name: Laura Lee
Date of Birth: March 10, 1982
Gender: Female
Relationship: Wife
Dependent Type: Adult

Child 1:
Full Name: Ethan Lee
Date of Birth: December 15, 2010
Gender: Male
Relationship: Son
Dependent Type: Child

Child 2:
Full Name: Ema Lee
Date of Birth: April 06, 2007
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Susan Wilson
Last Medical Checkup Date: July 10, 2023
Medical Conditions:
Type 2 Diabetes
Diagnosis Date: May 5, 2012
Treatment: Metformin and Dietary Management
High Cholesterol (Hypercholesterolemia)
Diagnosis Date: October 20, 2018
Treatment: Statins (Ato

In [31]:
## AFTER FINETUNING
test_data = ["""
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Rachel Turner
Date of Birth: November 8, 1990
Gender: Female
Address: 789 Birch Road, Harborview, CA
Phone Number: (555) 123-7890
Email: rachel.turner@email.com
Occupation: Software Developer

Dependents:
Spouse:
Full Name: Matthew Turner
Date of Birth: April 15, 1992
Gender: Male
Relationship: Husband
Dependent Type: Adult


Medical History:
Rachel Turner
Primary Care Physician: Dr. Emily Harris
Last Medical Checkup Date: August 15, 2023
Medical Conditions:
Anxiety Disorder
Diagnosis Date: June 5, 2015
Treatment: Cognitive-Behavioral Therapy and Medication (SSRIs)
Seasonal Affective Disorder (SAD)
Diagnosis Date: December 10, 2012
Treatment: Light Therapy and Lifestyle Management
Allergies: None

Medications:
Medication Name: Sertraline (for anxiety)
Dosage: 50 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Harborview General Hospital
Reason: None
Date: N/A

Surgeries:

Surgery Type: None

Matthew Turner
Primary Care Physician: Dr. Emily Harris
Last Medical Checkup Date: August 15, 2023
- No pre-existing medical conditions
- Hospitalized for pneumonia in 2016, fully recovered
- No chronic illnesses or disabilities ->:
"""]

sequences = pipeline(
    test_data,
    max_length=1024,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])

0 
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Rachel Turner
Date of Birth: November 8, 1990
Gender: Female
Address: 789 Birch Road, Harborview, CA
Phone Number: (555) 123-7890
Email: rachel.turner@email.com
Occupation: Software Developer

Dependents:
Spouse:
Full Name: Matthew Turner
Date of Birth: April 15, 1992
Gender: Male
Relationship: Husband
Dependent Type: Adult


Medical History:
Rachel Turner
Primary Care Physician: Dr. Emily Harris
Last Medical Checkup Date: August 15, 2023
Medical Conditions:
Anxiety Disorder
Diagnosis Date: June 5, 2015
Treatment: Cognitive-Behavioral Therapy and Medication (SSRIs)
Seasonal Affective Disorder (SAD)
Diagnosis Date: December 10, 2012
Treatment: Light Therapy and Lifestyle Management
Allergies: None

Medications:
Medication Name: Sertraline (for anxiety)
Dosage: 50 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Harbor

In [32]:
## AFTER FINETUNING
test_data = ["""
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Samuel Harris
Date of Birth: March 25, 1986
Gender: Male
Address: 456 Spruce Street, Woodville, TX
Phone Number: (555) 456-7891
Email: samuel.harris@email.com
Occupation: Accountant

Dependents:
Spouse:
Full Name: Olivia Harris
Date of Birth: October 20, 1988
Gender: Female
Relationship: Wife
Dependent Type: Adult

Child:
Full Name: Phoebe Harris
Date of Birth: December 12, 2012
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Samuel Harris
Primary Care Physician: Dr. Michael Turner
Last Medical Checkup Date: September 20, 2023
Medical Conditions:
Asthma
Diagnosis Date: January 5, 1995
Treatment: Inhalers (Albuterol and Advair)
GERD (Gastroesophageal Reflux Disease)
Diagnosis Date: April 15, 2010
Treatment: Proton Pump Inhibitors (PPIs)
Allergies:
Allergy: Shellfish
Severity: Moderate
Medications 12:

Medication Name: Albuterol (for asthma)
Dosage: As needed
Frequency: As needed
Medication Name: Advair (for asthma)
Dosage: As prescribed
Frequency: As prescribed
Medication Name: Omeprazole (for GERD)
Dosage: 20 mg daily
Frequency: Daily

Hospitalizations:
Hospital Name: Woodville Regional Hospital
Reason: Asthma
Date: February 22, 2015

Surgeries:
Surgery Type: None

Olivia Harris
Primary Care Physician: Dr. Michael Turner
Last Medical Checkup Date: March 18, 2023

Medical Conditions:
Anxiety Disorder
Diagnosis Date: June 5, 2015
Treatment: Cognitive-Behavioral Therapy and Medication (SSRIs)

Medications:
Medication Name: Sertraline (for anxiety)
Dosage: 50 mg daily
Frequency: Daily

Phoebe Harris:
- No major health conditions or surgeries in the past 5 years
- Prescription medications: None ->:
"""]

sequences = pipeline(
    test_data,
    max_length=1024,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])


0 
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Samuel Harris
Date of Birth: March 25, 1986
Gender: Male
Address: 456 Spruce Street, Woodville, TX
Phone Number: (555) 456-7891
Email: samuel.harris@email.com
Occupation: Accountant

Dependents:
Spouse:
Full Name: Olivia Harris
Date of Birth: October 20, 1988
Gender: Female
Relationship: Wife
Dependent Type: Adult

Child:
Full Name: Phoebe Harris
Date of Birth: December 12, 2012
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Samuel Harris
Primary Care Physician: Dr. Michael Turner
Last Medical Checkup Date: September 20, 2023
Medical Conditions:
Asthma
Diagnosis Date: January 5, 1995
Treatment: Inhalers (Albuterol and Advair)
GERD (Gastroesophageal Reflux Disease)
Diagnosis Date: April 15, 2010
Treatment: Proton Pump Inhibitors (PPIs)
Allergies:
Allergy: Shellfish
Severity: Moderate
Medications 12

In [39]:
# TRAINING DATA

test_data = ["""Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:

Name: John Smith
Age: 35
Gender: Male
Occupation: Software Engineer
Contact Information: Address: 123 Main Street, Anytown, USA

Phone: (555) 123-4567
Email: johnsmith@example.com

Dependents:
Spouse: Jane Smith (Age: 32)
Children:
Emily Smith (Age: 7)
Ethan Smith (Age: 4)

Medical History:
John Smith:
- No chronic illnesses or pre-existing conditions.
- Last visited a doctor for a routine check-up 6 months ago.
 Jane Smith:
 - No chronic illnesses or pre-existing conditions.
 - Last visited a doctor for a routine check-up 1 year ago.

 Emily Smith:
 - No chronic illnesses or pre-existing conditions.
 - Last visited a doctor for a routine check-up 3 months ago.
 Ethan Smith:
 - No chronic illnesses or pre-existing conditions.
 - Last visited a doctor for a routine check-up 6 months ago.
 ->:
""",
"""
Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:


Name: John Smith
Date of Birth: 01/15/1980
Gender: Male
Occupation: Software Engineer
Contact Information:
- Address: 123 Main Street, Anytown, USA
- Phone: (555) 123-4567
- Email: johnsmith@email.com

Dependents:
Spouse: Jane Smith
Date of Birth: 04/20/1982
Gender: Female
Occupation: Teacher

Medical History:
John Smith:
- No known allergies
- Past medical conditions: None
- Past surgeries: Appendectomy in 2005
Jane Smith:
- Allergies: Penicillin
- Past medical conditions: Asthma
- Past surgeries: None
->:
"""]

sequences = pipeline(
    test_data,
    max_length=1024,  #200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)

for ix,seq in enumerate(sequences):
    print(ix,seq[0]['generated_text'])

0 Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below: 

Name: John Smith
Age: 35
Gender: Male
Occupation: Software Engineer
Contact Information: Address: 123 Main Street, Anytown, USA

Phone: (555) 123-4567                      
Email: johnsmith@example.com

Dependents:
Spouse: Jane Smith (Age: 32)
Children: 
Emily Smith (Age: 7)
Ethan Smith (Age: 4)

Medical History:
John Smith:
- No chronic illnesses or pre-existing conditions.
- Last visited a doctor for a routine check-up 6 months ago.
 Jane Smith:
 - No chronic illnesses or pre-existing conditions.
 - Last visited a doctor for a routine check-up 1 year ago.
 
 Emily Smith:
 - No chronic illnesses or pre-existing conditions.
 - Last visited a doctor for a routine check-up 3 months ago.
 Ethan Smith:
 - No chronic illnesses or pre-existing conditions.
 - Last visited a doctor for a routine check-up 6 months ago.
 ->:

 ->:

Coverage Detail

**INFERENCE THE MODEL BY DOWNLOADING IT**

In [34]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer

base_model_name="TinyPixel/Llama-2-7B-bf16-sharded"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, "/content")

RuntimeError: ignored

In [ ]:
eval_prompt = """Act as an Insurance underwriting expert for ABC Health Care and give health insurance underwriting as per applicant's information mentioned below:
Full Name: Michelle Anderson
Date of Birth: June 5, 1970
Gender: Female
Address: 123 Oakwood Drive, Green Hills, CA
Phone Number: (555) 123-4567
Email: michelle.anderson@email.com
Occupation: Registered Nurse

Dependents:
Spouse:
Full Name: David Anderson
Date of Birth: August 15, 1968
Gender: Male
Relationship: Husband
Dependent Type: Adult

Child:
Full Name: Emily Anderson
Date of Birth: November 20, 1995
Gender: Female
Relationship: Daughter
Dependent Type: Child

Medical History:
Primary Care Physician: Dr. Rachel Smith
Last Medical Checkup Date: August 5, 2023
Medical Conditions:
Hypertension
Diagnosis Date: January 5, 2010
Treatment: Medication (Lisinopril)
Asthma
Diagnosis Date: August 15, 2000
Treatment: Inhalers (Albuterol and Advair)
Allergies: None

Medications:
Medication Name: Lisinopril (for hypertension)
Dosage: 10 mg daily
Frequency: Daily
Medication Name: Albuterol (for asthma)
Dosage: As needed
Frequency: As needed
Medication Name: Advair (for asthma)
Dosage: As prescribed
Frequency: As prescribed

Hospitalizations:
Hospital Name: Green Hills General Hospital
Reason: Appendectomy
Date: March 20, 1992

Surgeries:

Surgery Type: Appendectomy
Date: March 20, 1992
->:
"""
model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))